# Profit-seeking Algorithm

In [ ]:
import pandas as pd
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from src.utils.bmeapi import APIBMEHandler

In [ ]:
class EnvData():
    
    def __init__(self):
        api = APIBMEHandler('IBEX', 'rl-algo')
        self.close_data = api.get_close_data()
        self.bench = api.get_close_data_ticker('benchmark')
        self.first_year = self.bench.index[0].year
        self.last_year = self.bench.index[-1].year

    @staticmethod
    def median_sharpe(series):
        return series.median() / series.std()

    def generate_data(self, start: int = None, end: int = None):
        if start is None:
            start = self.first_year

        if end is None:
            end = self.last_year

        if start < self.first_year or start > end:
            raise ValueError(
                f"Invalid start {action}"
            )
        if end > self.last_year:
            raise ValueError(
                f"Invalid end {action}"
            )

        data = self.close_data.loc[str(start):str(end), :].dropna(how='all', axis=1)
        returns = (data.diff() / data.shift(1)).dropna(how='all')
        bench_ret = (self.bench.diff() / self.bench.shift(1)).dropna(how='all')
        median_returns = returns.median(axis=1)
        rank_sh = {}
        for win in [10, 30, 60]:
            sh = returns.rolling(win).apply(self.median_sharpe)
            rank_sh_df = sh.rank(axis=1) 
            rank_sh_df = rank_sh_df.divide(rank_sh_df.max(axis=1), axis=0).dropna(how='all')
            rank_sh[win] = rank_sh_df
        
        env_data = {}
        for ticker in returns.columns:
            df = pd.DataFrame(returns[ticker].rename('returns'))
            for win in rank_sh.keys():
                df[f'sh_{win}'] = pd.Series(rank_sh[win][ticker])
            df['median_ret'] = median_returns
            df['bench_ret'] = bench_ret
            df['position'] = 0
            df.dropna(inplace=True)
            if len(df) >= 50:
                env_data[ticker] = df

        return env_data

In [ ]:
class AlgoEnv:
    def __init__(self, data):
        
        self.data = data
        
        self.actions_fn = {
            0: self._maintain,
            1: self._buy,
            2: self._sell,
        }
        self.n_actions = len(self.actions_fn)
        self.n_features = 7
        self.tickers = list(self.data.keys())

        self.ticker = None
        self.episode = None
        self.len_episode = None
        self.position = 0
        self.action_hist = None
        self.idx = 0

    def _reset_params(self):
        self.ticker = None
        self.episode = None
        self.len_episode = None
        self.position = 0
        self.action_hist = None
        self.idx = 0
        
    def _compute_episode(self):
        self.ticker = random.choice(self.tickers)
        high = len(self.data[self.ticker]) - 30
        init_pos = np.random.randint(low=0, high=high)
        self.episode = self.data[self.ticker][init_pos:]
        self.episode['position'].values[:] = 0
        self.len_episode = len(self.episode)
        self.action_hist = np.zeros(self.len_episode)
            
    def reset(self):
        
        self._reset_params()
        self._compute_episode()
        
        return self._get_new_observation()
    
    def _get_new_observation(self):
        return self.episode.values[self.idx]
    
    def _compute_alpha(self, lag=1):
        if self.position == 0:
            return 0
        algo_ret = self.episode['returns'].values[self.idx + lag]
        bench_ret = self.episode['bench_ret'].values[self.idx + lag]
        alpha = np.log(algo_ret - bench_ret + 1)
        return alpha

    def _maintain(self):
        alpha = self._compute_alpha()
        return alpha, False
    
    def _buy(self):
        alpha = self._compute_alpha()
        if self.position == 0:
            self.position = 1
            self.episode['position'].values[self.idx + 1:] = self.position

        return alpha, False
    
    def _sell(self):
        alpha = self._compute_alpha()
        if self.position == 1:
            self.position == 0
            self.episode['position'].values[self.idx + 1:] = self.position
            alpha += self._compute_alpha(lag=2)
            return alpha, True

        return alpha, False
    
    def step(self, action):
        
        act_fn = self.actions_fn.get(action)
        if act_fn is None:
            raise ValueError(
                f"Invalid action {action}. Valid actions {actions_fn.keys()}"
            )
        
        self.action_hist[self.idx] = action
        
        reward, done = act_fn()
        
        self.idx += 1
        if self.idx + 2 == self.len_episode:
            done = True

        return self._get_new_observation(), reward, done, {}
    
    def action_sample(self):
        return np.random.choice([0, 1, 2], p=[0.8, 0.1, 0.1])
        
        

In [ ]:
class DDQNAgent():

    def __init__(
        self, env, train=True, gamma=0.99, epsilon=0.9, alpha=0.01, 
        batch_size=64, buffer_size=50000,
        hidden_neurons=32
    ):
        
        self.env = env
        self.na = env.n_actions
        self.n_features = env.n_features
        self.train = train
        self.epsilon = epsilon 
        self.gamma = gamma 
        self.minus_factor = None
        self.div_factor = None

        self.batch_size = batch_size
        self.buffer_size = buffer_size
        self.buffer = deque(maxlen=self.buffer_size)
        self.hidden_neurons = hidden_neurons
        
        self.primary_model = self.network()
        self.primary_model.compile(loss="mse", optimizer=Adam(learning_rate=alpha))

        self.target_model = self.network()
        self.update_target()
    
    def network(self):
        inp = Input(shape=(self.n_features, ))
        capa = Dense(int(self.hidden_neurons), activation="relu")(inp)
        capa = Dense(int(self.hidden_neurons), activation="relu")(inp)
        capa = Dense(int(self.hidden_neurons), activation="relu")(capa)
        out = Dense(self.na, activation="linear")(capa)
        
        return Model(inp, out)

    def experience(self, s, a, r, s1, done=False):
        if done:
            s1 = None
        self.buffer.append((s, a, r, s1))

    def sample(self):
        # CER
        idx = np.hstack([np.random.choice(
                            (len(self.buffer)-1),
                            size=self.batch_size - 1,
                            replace=False),(len(self.buffer)-1)])

        return [self.buffer[i] for i in idx]
    
    @staticmethod
    def getQs(s, model): # New parameter model
        return model(s).numpy()

    def act(self, s):
        if self.train:
            if np.random.rand() > self.epsilon:
                a = self._act_no_explore(s)
            else: 
                a = self.env.action_sample()
        else:
            a = self._act_no_explore(s)
        return a
    
    def _act_no_explore(self, s):
        q = self.getQs(
            s=s.reshape(1, -1),
            model=self.primary_model
        )
        return np.argmax(q)

    def getQ(self, s, a):
        return self.getQs(s)[:, a]

    # update target with primary model weights
    def update_target(self):
        self.target_model.set_weights(self.primary_model.get_weights())
            
    def learn(self):
        batch = self.sample()
        s_list = np.array([experience[0].tolist() for experience in batch])
        s1_list = np.array([
            experience[3].tolist() for experience in batch 
            if experience[3] is not None
        ])
        qs = self.getQs(s_list, self.primary_model)
        # Primary model selects the best next action...
        qs1 = self.getQs(s1_list, self.primary_model)
        # ...and target model evaluates it
        q_target = self.getQs(s1_list, self.target_model)
        targets = qs

        k = 0
        for i, (s, a, r, s1) in enumerate(batch):
            if s1 is None:
                targets[i][a] = r
            else:
                best_next_action = np.argmax(qs1[k])
                targets[i][a] = r + self.gamma * q_target[k, best_next_action]
                k += 1
                
        loss = self.primary_model.train_on_batch(s_list, targets)

        return loss

------------------
# Entrenamiento

#### Generate Data

In [ ]:
gen = EnvData()
train_data = gen.generate_data(2017, 2018)
test_data = gen.generate_data(2019, 2020)

#### Init Env & Agent

In [ ]:
# Agent Params
epsilon = 1
min_epsilon = 0.03
gamma = 0.995
alpha = 0.001
buffer_size = 5000
batch_size = 64
hidden_neurons = 64

# Training Params
nepisodes = 3000
n_log = 250
epsilon_decay = (epsilon - min_epsilon) / (nepisodes * 0.8)
learn_after = batch_size
update_target_after = 10 * learn_after

env = AlgoEnv(train_data)

agent = DDQNAgent(
    env, gamma=gamma, epsilon=epsilon, alpha=alpha,
    batch_size=batch_size, buffer_size=buffer_size,
    hidden_neurons=hidden_neurons
)

#### Collect Experiences

In [ ]:
s = env.reset()
for exps in range(buffer_size):   
    a = env.action_sample()
    s1, r, done, _ = env.step(a)
    agent.experience(s, a, r, s1, done)
    s = s1
    if not exps % 2500:
        print(f'buffer exps: {exps}')
    if done:
        s = env.reset()

#### Train Algo

In [ ]:
learn_counter = 0
history_steps = []
history_rewards = []
history_disc_rewards = []
history_losses = []

for episode in range(nepisodes):
    s = env.reset()
    step = 0
    cum_reward = 0
    dis_cum_reward = 0
    episode_losses = []
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        agent.experience(s, a, r, s1, done)
        learn_counter += 1
        cum_reward += r
        dis_cum_reward += agent.gamma ** step * r
        s = s1
        step += 1
        if not learn_counter % learn_after:
            agent.learn()
        if not learn_counter % update_target_after:
            agent.update_target()
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            history_rewards.append(cum_reward)
            history_disc_rewards.append(dis_cum_reward)
            history_steps.append(step)
            if not episode % n_log:
                agent.learn()
                print(
                    f'Episode: {episode}, '
                    f'steps: {np.round(np.mean(history_steps[-n_log:]), 2)}, '
                    f'rew: {np.round(np.mean(history_rewards[-n_log:]), 4)}, '
                    f'eps: {np.round(agent.epsilon, 2)}'
                )
            break

In [ ]:
pd.DataFrame(history_rewards).rolling(100).mean().plot()

In [ ]:
agent.train = False
hist_rewards = {
    'train': [],
    'test': []
}
for key in hist_rewards.keys():
    for _ in range(100):
        s = env.reset()
        cum_reward = 0
        step = 0
        while True:
            a = agent.act(s)
            s, r, done, _ = env.step(a)
            step += 1
            cum_reward += agent.gamma ** step * r
            if done:
                hist_rewards[key].append(cum_reward)
                break

In [ ]:
print(np.mean(hist_rewards['train']))
print(np.mean(hist_rewards['test']))

In [ ]:
data = [hist_rewards['test'], hist_rewards['train']]
fig = plt.figure(figsize =(7, 5))
ax = fig.add_subplot(111)
ax.set_yticklabels(['test', 'train'])
 
bp = ax.boxplot(data, vert=False, showfliers=False)

In [ ]:
ticker = env.episode[:env.idx + 2].returns + 1
ticker.values[0] = 100
ticker_nav = ticker.cumprod()
ticker_nav.plot()
bench = env.episode[:env.idx + 2].bench_ret + 1
bench.values[0] = 100
bench_nav = bench.cumprod()
bench_nav.plot()
position = env.episode[:env.idx + 2].position
position_s = position.shift(1)
position_s = ticker_nav[position_s == 1]
position_s.plot(color='red')